In [ ]:
from pathlib import Path
import pandas as pd
idx_slice = pd.IndexSlice
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
import plotly.graph_objects as go

import sys
sys.path.append('..')
from scripts.plot_helpers import (
    chdir_to_root_dir,
    read_stats_dict,
    read_csv_nafix,
    prepare_dataframe,
    nice_title,
    load_plot_config,
    get_scen_col_function,
    save_plotly_fig)


chdir_to_root_dir()

In [ ]:
yaml_config = load_plot_config()

RUN_NAME_PREFIX = yaml_config['run']['name_prefix']
SUMMARY_VERSION = yaml_config['run']['summary_version']
SDIR = Path.cwd() / "results" / f"{RUN_NAME_PREFIX}_summary_{SUMMARY_VERSION}"
SDIR.mkdir(exist_ok=True, parents=True)

COUNTRIES = yaml_config['data']['countries']
YEARS = yaml_config['data']['years']
INDEX_LEVELS_TO_DROP = yaml_config['data']['index_levels_to_drop']

IDX_GROUP = idx_slice[[RUN_NAME_PREFIX], :, COUNTRIES, YEARS]
IDX_GROUP_NAME = "".join(COUNTRIES) + "_MAIN"

print(f"Index Group Name: {IDX_GROUP_NAME}")
print(f"Index Group: {IDX_GROUP}")

# Scenario filtering
SCEN_FILTER = yaml_config['data']['scen_filter']
scen_order = yaml_config['data']['scen_order']

# Get scenario column function
scen_col_func_name = yaml_config['data']['scen_col_function']
set_scen_col = get_scen_col_function(scen_col_func_name)
print(f"Using scenario column function: {scen_col_func_name}")

# Plot dimensions
width = yaml_config['plot']['width']
heights = yaml_config['plot']['heights']

# Default figure kwargs
fig_kwargs = yaml_config['plot']['default_kwargs'].copy()
fig_kwargs['width'] = width
fig_kwargs['height'] = heights['medium']

print(f"\nConfiguration loaded successfully!")
print(f"Countries: {COUNTRIES}")
print(f"Years: {YEARS}")

In [ ]:
index_cols = [
    "run_name_prefix", "run_name", "country", "year", "simpl", "clusters", "ll", "opts", "sopts", "discountrate", "demand", "h2export"
 ]
marginal_prices = read_csv_nafix(SDIR / "marginal_prices.csv", index_col=index_cols)

# Filter for countries that actually exist in the data
available_countries = marginal_prices.index.get_level_values('country').unique().tolist()
missing_countries = [c for c in COUNTRIES if c not in available_countries]
filtered_countries = [c for c in COUNTRIES if c in available_countries]

if missing_countries:
    print(f"⚠️  Warning: Countries not found in data: {missing_countries}")
    print(f"Using available countries: {filtered_countries}")

# Update IDX_GROUP with filtered countries
IDX_GROUP_FILTERED = idx_slice[[RUN_NAME_PREFIX], :, filtered_countries, YEARS]

marginal_prices = prepare_dataframe(marginal_prices, IDX_GROUP_FILTERED, INDEX_LEVELS_TO_DROP, set_scen_col, drop_zero=False)
marginal_prices = marginal_prices.query("scen in @SCEN_FILTER") if SCEN_FILTER else marginal_prices


In [ ]:
SDIR

In [ ]:
df = marginal_prices.query("variable in ['H2 export bus']").copy()


## Marginal prices | H2 export price

In [ ]:
idx = pd.IndexSlice

df = marginal_prices[
    (marginal_prices['year'].isin([2035, 2050])) & 
    (marginal_prices['variable'] == 'H2 export bus')
].copy()

output_path = SDIR / "marginal_prices_prepared.csv"
df.to_csv(output_path)
print(f"Saved to: {output_path}")

In [ ]:
df

In [ ]:
def marginal_price_dumbbell_fig(df, year, data_start="low", data_mid="mid", data_end="high"):

    df = df[(df.year == year)].copy()
    countries = df["country"].unique()

    line_x, line_y = [], []
    vals_start, vals_mid, vals_end, valid = [], [], [], []
    skipped = []

    for country in countries:
        s = df.loc[(df.scen == f"{country}-{data_start}") & (df.country == country), "value"]
        m = df.loc[(df.scen == f"{country}-{data_mid}") & (df.country == country), "value"]
        e = df.loc[(df.scen == f"{country}-{data_end}") & (df.country == country), "value"]
        if len(s) > 0 and len(e) > 0:
            vs, ve = s.values[0], e.values[0]
            vals_start.append(vs)
            vals_end.append(ve)
            vals_mid.append(m.values[0] if len(m) > 0 else None)
            line_x.extend([vs, ve, None])
            line_y.extend([country, country, None])
            valid.append(country)
        else:
            missing = []
            if len(s) == 0:
                missing.append(data_start)
            if len(e) == 0:
                missing.append(data_end)
            skipped.append(f"{country} (missing: {', '.join(missing)})")

    fig = go.Figure(data=[
        go.Scatter(x=line_x, y=line_y, mode="lines", showlegend=False,
                   marker=dict(color="grey")),
        go.Scatter(
            x=vals_start, y=valid, mode="markers+text", name=data_start,
            text=[f"{v:.1f}" for v in vals_start], textposition="middle left",
            textfont=dict(size=11), marker=dict(color="#99bdcc", size=13),
        ),
        go.Scatter(
            x=[v for v in vals_mid if v is not None],
            y=[c for c, v in zip(valid, vals_mid) if v is not None],
            mode="markers", name=data_mid,
            marker=dict(color="#669db2 ", size=13),
        ),
        go.Scatter(
            x=vals_end, y=valid, mode="markers+text", name=data_end,
            text=[f"{v:.1f}" for v in vals_end], textposition="middle right",
            textfont=dict(size=11), marker=dict(color="#005b7f", size=13), #669db2 
        ),
    ])

    fig.update_layout(
        title=dict(text=nice_title(
            f"Marginal price for H2 at export port in {year}",
            "Per country and H2 export volume in EUR/MWh_H2_LHV",
        )),
        height=max(500, len(valid) * 45),
        legend_itemclick=False,
    )
    return fig, skipped


In [ ]:
fig, skipped = marginal_price_dumbbell_fig(df, year=2050, data_start="low", data_end="high")
if skipped:
    print(f"Skipped: {', '.join(skipped)}")
fig.show()


In [ ]:
fig, skipped = marginal_price_dumbbell_fig(df, year=2035, data_start="low", data_end="high")
if skipped:
    print(f"Skipped: {', '.join(skipped)}")
fig.show()
